## Nature 论文解析任务 - Task 2
_本 Notebook 用于解析 Nature 搜索结果并生成结构化数据_


In [2]:
# 导入所需库
from bs4 import BeautifulSoup  # HTML 解析库
import json  # JSON 处理
import re    # 正则表达式
from pprint import pprint  # 美化输出

### 步骤 1：加载本地页面内容


In [3]:
# 读取保存的 HTML 文件
with open('page1.txt', 'r', encoding='utf-8') as f:
    html = f.read()

# 初始化 BeautifulSoup 解析器（使用 lxml 解析引擎）
# 如果没有安装 lxml：!pip install lxml
soup = BeautifulSoup(html, 'html.parser')

print("HTML 内容加载完成，文档大小：", len(html), "字符")

HTML 内容加载完成，文档大小： 340494 字符


### 步骤 2：解析论文数据


In [4]:
# 初始化存储容器
journal_data = []  # 按期刊分类的论文数据

# 使用 CSS 选择器定位所有论文条目
articles = soup.select('li.app-article-list-row__item article.c-card')
print(f"共找到 {len(articles)} 篇论文")


共找到 50 篇论文


#### 单篇论文解析流程


In [5]:
for idx, article in enumerate(articles, 1):
    # 提取标题和链接
    title_tag = article.find('h3', class_='c-card__title').find('a')
    title = title_tag.text.strip() if title_tag else "No title"
    url = title_tag['href'] if title_tag else "No URL"
    
    # 处理相对链接（如果需要绝对地址）
    if url and not url.startswith('http'):
        url = f"https://www.nature.com{url}"
    
    # 提取作者列表
    authors = [author.text.strip() 
              for author in article.select('[itemprop="creator"] [itemprop="name"]')]
    
    # 提取摘要描述
    desc_tag = article.find('div', {'data-test': 'article-description'})
    description = desc_tag.text.strip() if desc_tag else "no description"
    
    # 提取元信息部分
    meta_section = article.find('div', class_='c-meta')
    
    # 解析文章类型
    article_type = meta_section.find('span', class_='c-meta__type').text.strip() if meta_section else "Unknown"
    
    # 解析期刊名称
    journal_tag = meta_section.find('div', {'data-test': 'journal-title-and-link'})
    journal_name = journal_tag.text.strip() if journal_tag else "Unknown"
    
    # 解析卷号和页码
    volume_tag = meta_section.find('div', {'data-test': 'volume-and-page-info'})
    volume_page_info = volume_tag.text.strip() if volume_tag else ""

    # 按期刊名称进行分类存储
    target_journal = next(
        (j for j in journal_data if j['journal'] == journal_name),
        None
    )

    # 如果该期刊不存在于列表中，则新建条目
    if not target_journal:
        target_journal = {
            "journal": journal_name,
            "papers": []
        }
        journal_data.append(target_journal)

    # 构建论文条目字典
    paper_entry = {
        "title": title,
        "url": url,
        "authors": authors,
        "description": description,
        "type": article_type,
        "volume_page_info": volume_page_info
    }
    
    target_journal['papers'].append(paper_entry)

    # 打印进度（每5篇输出一次）
    if idx % 5 == 0:
        print(f"已处理 {idx}/{len(articles)} 篇论文")

已处理 5/50 篇论文
已处理 10/50 篇论文
已处理 15/50 篇论文
已处理 20/50 篇论文
已处理 25/50 篇论文
已处理 30/50 篇论文
已处理 35/50 篇论文
已处理 40/50 篇论文
已处理 45/50 篇论文
已处理 50/50 篇论文


### 步骤 3：统计与输出


In [6]:
# 生成期刊论文数量统计
journal_counts = {j['journal']: len(j['papers']) for j in journal_data}

print("各期刊论文数量统计：")
for journal, count in journal_counts.items():
    print(f"· {journal.ljust(25)}: {str(count).rjust(3)} 篇")

# 显示样例数据结构
print("\n数据结构示例：")
pprint(journal_data[0]['papers'][0] if journal_data else {})

各期刊论文数量统计：
· Nature Machine Intelligence:   6 篇
· Nature Communications    :  10 篇
· Scientific Reports       :  12 篇
· Nature Methods           :   1 篇
· npj Digital Medicine     :   3 篇
· Nature Medicine          :   2 篇
· Nature                   :   4 篇
· Nature Human Behaviour   :   2 篇
· npj Precision Oncology   :   1 篇
· Nature Computational Science:   2 篇
· BDJ Open                 :   1 篇
· Nature Reviews Urology   :   1 篇
· Communications Materials :   1 篇
· Humanities and Social Sciences Communications:   1 篇
· npj Biodiversity         :   1 篇
· Eye                      :   1 篇
· npj Computational Materials:   1 篇

数据结构示例：
{'authors': ['Jianing Qiu', 'Kyle Lam', 'Eric J. Topol'],
 'description': 'Large language model-based agentic systems can process input '
                'information, plan and decide, recall and reflect, interact '
                'and collaborate, leverage various tools and act. This opens '
                'up a wealth of opportunities within medicine a

### 步骤 4：保存结构化数据


In [8]:
# 将数据保存为 JSON 文件
with open('nature_llm_pre.json', 'w', encoding='utf-8') as f:
    json.dump(journal_data, f, 
             indent=2,  # 保持缩进
             ensure_ascii=False)  # 允许显示非 ASCII 字符

print("\n保存成功！文件已保存为 nature_llm_pre.json")


保存成功！文件已保存为 nature_llm_pre.json
